In [3]:
from embedder import Embedder
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
import numpy as np

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
!uv run python download.py



  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [8]:
### Q1. Embedding a query
q  = "How does approximate nearest neighbor search work?"
v = model.encode(q)
v[0]

np.float32(-0.020582005)

### Q2. Cosine similarity


In [9]:
## Q2, Q3

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]



In [10]:
# Q2. Cosine similarity
documents = ["02-vector-search/lessons/07-sqlitesearch-vector.md"]

texts = [doc for doc in documents]
v = model.encode(texts)

q = model.encode("How does approximate nearest neighbor search work?")

scores = v.dot(q)
scores

array([0.3770523], dtype=float32)

### Q3. Chunking and search by hand

In [11]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]


from embedder import Embedder
embedder = Embedder()
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
texts = [chunk['content'] for chunk in chunks]
X = embedder.encode_batch(texts)

v = embedder.encode( "How does approximate nearest neighbor search work?")
scores = X.dot(v)
best_idx = scores.argmax()
print(scores[best_idx])
print(chunks[best_idx]['filename'])

0.6489016216319834
02-vector-search/lessons/07-sqlitesearch-vector.md


In [12]:
# query = "What metric do we use to evaluate a search engine?"
# v_query = model.encode(query)
# scores = X.dot(v_query)
# best_idx = scores.argmax()
# print(scores[best_idx])
# print(chunks[best_idx]['filename'])


In [13]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)
query = "What metric do we use to evaluate a search engine?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

## Q5. Text search vs vector search

In [14]:
#Index the same chunks with Index from minsearch. Use content as a text field.(Text search)
from minsearch import Index

text_index = Index(
    text_fields=["content"],
)
text_index.fit(chunks)

#text search
query = "How do I store vectors in PostgreSQL?"
text_results = text_index.search(query, num_results=5)
text_results[0]
print(chunks[best_idx]['filename'])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [15]:
## Vector search
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)
query = "How do I store vectors in PosrgreSQL?"
query_vector = model.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
vector_results[0]


{'start': 0,
 'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWORD

## Q6. Hybrid search

In [16]:
#Index the same chunks with Index from minsearch. Use content as a text field.(Text search)
from minsearch import Index

text_index = Index(
    text_fields=["content"],
)
text_index.fit(chunks)

#text search
query = "How do I give the model access to tools?"
text_results = text_index.search(query, num_results=5)
text_results[0]
print(chunks[best_idx]['filename'])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [17]:
## Vector search
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)
query = "How do I give the model access to tools?"
query_vector = model.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
vector_results[0]

{'start': 2000,
 'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline agen

In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [19]:
results = rrf([vector_results, text_results])
# results
print(results[0]['filename'])

01-agentic-rag/lessons/13-function-calling.md
